---
title: 树
date: 2026-11-26
category: notes
subject: python
summary: 树相关知识及题目、代码
---
           

## 1. 常见递归统计

### 计算一般树高度

```python
def height(node):
    if not node.children:
        return 0
    return 1 + max(height(child) for child in node.children)
```
### 计算二叉树高度

```python
def height(node):
    if not node :
        # 这里不能直接写not node.left and not node.right
        # 否则当node只有left时，访问height(node.right)就会因为NoneType
        # 没有right导致报错
        return -1
    return 1+max(height(node.right),height(node.right))

```

### 计算二叉树节点总数

```python
def count_nodes(node):
    if not node:
        return 0
    return 1 + count_nodes(node.left) + count_nodes(node.right)
```


### 计算二叉树叶子数

```python
def count_leaves(node):
    if not node:
        return 0
    if not node.left and not node.right:
        return 1
    return count_leaves(node.left) + count_leaves(node.right)
```

错误写法是先访问 `node.left`，再判断空树。空树传入时会发生：

```python
None.left
```

直接报 `AttributeError`。

---

## 2. 深度优先遍历  DFS 递归版


### 前序遍历

```python
def preorder(node):
    if not node:
        return
    print(node.val, end=' ')
    preorder(node.left)
    preorder(node.right)
```

### 中序遍历

```python
def inorder(node):
    if not node:
        return
    inorder(node.left)
    print(node.val, end=' ')
    inorder(node.right)
```

### 后序遍历

```python
def postorder(node):
    if not node:
        return
    postorder(node.left)
    postorder(node.right)
    print(node.val, end=' ')
```

---

## 3. 遍历背后的栈和队列

前序、中序、后序都是 DFS，递归版本使用 Python 的调用栈。

也可以用显式栈模拟。

比如前序：

```python
def preorder_stack(root):
    if not root:
        return

    stack = [root]

    while stack:
        node = stack.pop()
        print(node.val, end=' ')

        if node.right:
            stack.append(node.right)
        if node.left:
            stack.append(node.left)
```

注意先压右、再压左，这样弹出时才是先左后右。

中序显式栈写法：

```python
def inorder_stack(root):
    stack = []
    node = root

    while stack or node:
        while node:
            stack.append(node)
            node = node.left

        node = stack.pop()
        print(node.val, end=' ')
        node = node.right
```

后序显式栈常用双栈：

```python
def postorder_stack(root):
    if not root:
        return

    s1 = [root]
    s2 = []

    while s1:
        node = s1.pop()
        s2.append(node)

        if node.left:
            s1.append(node.left)
        if node.right:
            s1.append(node.right)

    while s2:
        print(s2.pop().val, end=' ')
```

---

## 4. 层序遍历  BFS

层序遍历是 BFS，按层从左到右访问，用队列。

```python
from collections import deque

def level_order(root):
    if not root:
        return

    q = deque([root])

    while q:
        node = q.popleft()
        print(node.val, end=' ')

        if node.left:
            q.append(node.left)
        if node.right:
            q.append(node.right)
```


---

## 5. 表达式树  AST

表达式也可以画成二叉树。

例如 `(3 + 4) * 2`：

```text
      *
     / \
    +   2
   / \
  3   4
```

三种遍历对应：

| 遍历 | 表达式形式 | 结果 |
|---|---|---|
| 前序 | 前缀表达式 | `* + 3 4 2` |
| 中序 | 中缀表达式 | `3 + 4 * 2` |
| 后序 | 后缀表达式 | `3 4 + 2 *` |

中序遍历表达式树时，如果去掉括号，运算优先级可能不明确，所以中缀表达式需要括号才严谨。

---

## 6. 二叉搜索树 BST

BST 是 Binary Search Tree，二叉搜索树。

性质：

> 对任意节点：左子树所有值 < 当前节点 < 右子树所有值，并且左右子树也分别满足 BST 性质。

注意是“整棵左子树”和“整棵右子树”，**不是只比较直接左右孩子**。

例如：

```text
      50
     /  \
   30    70
  / \    / \
20  40  60  80
```

这是合法 BST。

---

## 7. BST 查找

```python
def search(root, val):
    if not root:
        return False

    if root.val == val:
        return True

    if val < root.val:
        return search(root.left, val)
    else:
        return search(root.right, val)
```

查找 `60` 的路径：

```text
50 -> 70 -> 60
```

每次比较排除一半，平均复杂度是：

```text
O(log n)
```

---

## 8. BST 插入

```python
def insert(root, value):
    if not root:
        return TreeNode(value)

    if value < root.value:
        root.left = insert(root.left, value)
    elif value > root.value:
        root.right = insert(root.right, value)

    return root
```

插入的本质：

1. 按查找规则往下走。
2. 找到空位 `None`。
3. 创建新节点。
4. 把新节点接回父节点。

为什么必须写：

```python
root.left = insert(root.left, value)
```

因为 `root.left` 可能原来是 `None`，`insert()` 返回新建节点后要把它接回父节点。

同样，构建整棵树时要写：

```python
root = insert(root, value)
```

不能只写：

```python
insert(root, value)
```

否则第一次插入的新根会丢失。

**一定要有东西来接收返回值！！！**

---

## 9. BST 删除（待讲待查）

删除分三种情况。

| 情况 | 特征 | 操作 |
|---|---|---|
| 叶子节点 | 左右都为空 | 直接删除 |
| 只有一个孩子 | 只有左孩子或只有右孩子 | 用孩子替代自己 |
| 有两个孩子 | 左右都有 | 用后继节点替换，再删除后继 |

### 后继节点

后继是右子树中的最小值。

```python
def find_successor(node):
    succ = node.right
    while succ.left:
        succ = succ.left
    return succ
```

删除代码：

```python
def delete(root, val):
    if not root:
        return None

    if val < root.val:
        root.left = delete(root.left, val)
    elif val > root.val:
        root.right = delete(root.right, val)
    else:
        if not root.left:
            return root.right
        if not root.right:
            return root.left

        succ = root.right
        while succ.left:
            succ = succ.left

        root.val = succ.val
        root.right = delete(root.right, succ.val)

    return root
```

例如删除：

```text
      50
     /  \
   30    70
  / \    / \
20  40  60  80
```

删除 `50`：

1. `50` 有两个孩子。
2. 右子树是 `70, 60, 80`。
3. 右子树最小值是 `60`。
4. 用 `60` 替换 `50`。
5. 再删除右子树中的 `60`。

错误做法是直接用左孩子或右孩子替代，这样可能破坏 BST 性质。


---

## 10. 建树(普通二叉树)
> 若要建立BST二叉树，用insert进行创建
```python
root=None
for v in values:
    root=insert(root,v)
```



In [ ]:
from collections import deque

class TreeNode:
    def __init__(self,value):
        self.value=value
        self.left=None
        self.right=None

def tree_building(values): 
    # values为传入的list 如：1 2 3 4 5 -1 -1 -1 -1 -1
    if not values or values[0]==-1 :
        return None
    root=TreeNode(values[0])
    q=deque[root]
    i=1
    while q and i<len(values):
        node=q.popleft()
        if i<len(values) and values[i]!=-1:
            node.left=TreeNode(values[i])
            q.append(node.left)
        i+=1
        if i<len(values) and values[i]!=-1:
                    node.right=TreeNode(values[i])
                    q.append(node.right)
        i+=1
    return root

---

## 11. 判断BST（二分搜索树）是否合法 （FIP-W11-11D02）

In [ ]:
from collections import deque

class TreeNode:
    def __init__(self,value):
        self.value=value
        self.left=None
        self.right=None

def tree_building(values):
    if not values or values[0] == -1:
        return None

    root = TreeNode(values[0])
    q = deque([root])
    i = 1

    while q and i < len(values):
        node = q.popleft()

        if i < len(values) and values[i] != -1:
            node.left = TreeNode(values[i])
            q.append(node.left)
        i += 1

        if i < len(values) and values[i] != -1:
            node.right = TreeNode(values[i])
            q.append(node.right)
        i += 1

    return root


def is_valid(node, low=float('-inf'), high=float('inf')):
    if not node:
        return True

    left = node.left is None or node.left.value < node.value
    right = node.right is None or node.right.value > node.value

    if not left or not right:
        return False

    return is_valid(node.left, low, node.value) and \
           is_valid(node.right, node.value, high)

"""
1. `node.left`、`node.right` 可能为 `None`，直接取 `.value` 会报错，
   所以比较前要先判断它们是否存在。
2. 只判断 `node.left.value < node.value < node.right.value` 也不可取，
   有可能子树本身成立，但整棵树不成立。
   比如右子树里某个深层节点可能小于整棵树的根，只看父子关系发现不了。
        10
        /  \
     5      20
           /
          8
3. 修改方法是用上下界递归：每个节点额外带上允许的取值范围 `low` 和 `high`。
   进入左子树时把上界更新为当前节点的值，进入右子树时把下界更新为当前节点的值，
   这样任何一个深层节点都必须落在从根到它这条路径约束出的范围里。
"""

---

## 12. 查找路径（FIP-W11-11D01）

### 题目描述
废土档案按 `BST` 规则插入编号：小于当前结点去左子树，大于去右子树。查找目标时，记录经过的结点值（含最后访问的结点）。

请实现类 `BST`。插入只更新对象自己的 `self.root`，调用者不用再写 `root = insert(...)`。

若找到目标，`search_path` 返回完整路径（含目标值）；
若未找到，返回已访问路径并在末尾追加 -1。

### 类接口定义：
```python

class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):
        ...

    def search_path(self, target):
        ...

```
insert 把 val 插进这棵树。search_path 返回整数列表。

### 裁判测试程序样例：
```python

# 请在这里填写答案
#class BST:


n = int(input())
values = list(map(int, input().split()))
target = int(input())
tree = BST()
for v in values:
    tree.insert(v)
print(*tree.search_path(target))

```
输入样例:
```
7
50 30 70 20 40 60 80
40
```
输出样例:
```
50 30 40
```
未找到样例:
输入 `65` 时输出：

```50 70 60 -1```

In [ ]:
class TreeNode:
    def __init__(self,value):
        self.value=value
        self.left=None
        self.right=None
    """ 
    也可以将TreeNode定义放在BST类下，调用时只需改成BST.TreeNode()
    至于为什么要class BST后还要class TreeNode：
        `BST` 定义了一个类为树，而 `TreeNode` 则是定义一个类为节点，二者其实是两种东西。

    """

class BST:
    def __init__(self):
        self.root = None

    def insert(self, val):
        # `insert` 函数里有 `self` 后不能像往常一样直接递归写，
        # 这里需要先令 `node = self.root`，再进行循环判断；
        # 注意每次 base case，取 `left` 或 `right` 前也要判断是否为空。

        if not self.root:
            self.root=TreeNode(val)
            return
        node=self.root
        while True:
            if val>node.value:
                if not node.right:
                    node.right=TreeNode(val)
                    return
                node=node.right
            else:
                if not node.left:
                    node.left=TreeNode(val)
                    return
                node=node.left
        

    def search_path(self, target):
        path=[]
        node=self.root

        while node:
            path.append(node.value)
            if target== node.value:
                return path
            elif target> node.value:
                node=node.right
            else:
                node=node.left
        return path+[-1] # 能进行到这里说明没有找到